# 🧠 Sistem Deteksi Stres Karyawan — CC26-PSU196
## NOTEBOOK LENGKAP (Training + Testing dalam 1 File)

### 📌 Cara Pakai:
1. Upload file ini ke Google Colab
2. Upload juga `clean_dataset.csv` ke Colab
3. Jalankan semua cell dari atas ke bawah (**Runtime → Run All**)
4. Selesai! Hasil diagnosis akan muncul di bawah

---

# BAGIAN 1 — INSTALL & IMPORT
> Jalankan ini pertama kali

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print("✅ Semua library berhasil diinstall")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, json, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print("✅ Semua library berhasil diimport")

# BAGIAN 2 — CF EXPERT SYSTEM
> Ini adalah sistem pakar berbasis aturan (tidak perlu training data)
> Langsung bisa dipakai setelah cell ini dijalankan

In [ ]:
# ─── KNOWLEDGE BASE (Basis Pengetahuan dari Pakar) ───
KNOWLEDGE_BASE = {
    # Beban dan Tekanan Kerja
    "G1":  {"D2": 1.0},
    "G2":  {"D2": 1.0, "D3": 1.0},
    "G3":  {"D2": 1.0, "D3": 1.0},
    "G4":  {"D2": 0.8, "D3": 0.8},
    "G5":  {"D2": 1.0},
    "G6":  {"D2": 0.8, "D3": 0.8},
    "G7":  {"D1": 0.4, "D2": 0.4},
    "G8":  {"D1": 0.8, "D2": 0.8},
    "G9":  {"D2": 0.8, "D3": 0.8},
    "G10": {"D2": 0.9},
    # Konflik Peran
    "G11": {"D1": 0.5, "D2": 0.5},
    "G12": {"D2": 0.6, "D3": 0.6},
    "G13": {"D2": 0.8, "D3": 0.8},
    "G14": {"D2": 0.6, "D3": 0.6},
    "G15": {"D2": 0.8, "D3": 0.8},
    "G16": {"D2": 1.0, "D3": 1.0},
    "G17": {"D2": 1.0, "D3": 1.0},
    # Hubungan Interpersonal
    "G18": {"D1": 1.0},
    "G19": {"D1": 0.8, "D2": 0.8, "D3": 0.8},
    "G20": {"D2": 0.7, "D3": 0.7},
    "G21": {"D1": 0.8, "D2": 0.8, "D3": 0.8},
    "G22": {"D2": 0.8, "D3": 0.8},
    # Kejelasan Peran
    "G23": {"D2": 1.0},
    "G24": {"D1": 0.6, "D2": 0.6},
    "G25": {"D1": 0.5, "D2": 0.5, "D3": 0.5},
    "G26": {"D2": 1.0, "D3": 1.0},
    "G27": {"D1": 0.7, "D2": 0.7},
    "G28": {"D2": 0.8, "D3": 0.8},
    "G29": {"D2": 0.8},
    # Kepemimpinan
    "G30": {"D2": 0.4, "D3": 0.4, "D4": 0.4},
    "G31": {"D2": 0.7, "D3": 0.7},
    "G32": {"D2": 0.8, "D3": 0.8},
    "G33": {"D1": 0.9, "D2": 0.9, "D3": 0.9},
    "G34": {"D2": 0.6, "D3": 0.6},
    "G35": {"D2": 0.9, "D3": 0.9},
    "G36": {"D1": 0.8, "D2": 0.8, "D3": 0.8},
    # Pengembangan Karir
    "G37": {"D2": 0.9},
    "G38": {"D2": 1.0},
    "G39": {"D2": 0.4, "D4": 0.4},
    "G40": {"D1": 0.4, "D3": 0.4},
    "G41": {"D1": 0.6, "D2": 0.6},
    "G42": {"D2": 0.8, "D3": 0.8},
    "G43": {"D2": 0.4, "D4": 0.4},
}

DIAGNOSES = {
    "D1": "Tidak Stres",
    "D2": "Stres Ringan",
    "D3": "Stres Sedang",
    "D4": "Stres Berat",
}

USER_CF_MAP = {
    "tidak_pernah":  0.0,
    "jarang":        0.25,
    "kadang_kadang": 0.5,
    "sering":        0.75,
    "selalu":        1.0,
}

print("✅ Knowledge Base loaded!")
print(f"   Total gejala : {len(KNOWLEDGE_BASE)}")
print(f"   Diagnosis    : {DIAGNOSES}")

In [ ]:
# ─── FUNGSI CF ───

def cf_combine(cf_old, cf_new):
    if cf_old >= 0 and cf_new >= 0:
        return cf_old + cf_new * (1 - cf_old)
    elif cf_old < 0 and cf_new < 0:
        return cf_old + cf_new * (1 + cf_old)
    else:
        return (cf_old + cf_new) / (1 - min(abs(cf_old), abs(cf_new)))

def diagnose_cf(symptom_answers):
    """
    Input  : dict {kode_gejala: nilai_cf (0.0-1.0)}
    Output : hasil diagnosis lengkap
    """
    cf_scores = {d: 0.0 for d in DIAGNOSES}

    for gejala, cf_user in symptom_answers.items():
        if gejala not in KNOWLEDGE_BASE or cf_user == 0.0:
            continue
        for diag, cf_pakar in KNOWLEDGE_BASE[gejala].items():
            cf_g = cf_pakar * cf_user
            cf_scores[diag] = cf_combine(cf_scores[diag], cf_g)

    max_diag  = max(cf_scores, key=cf_scores.get)
    max_score = cf_scores[max_diag]

    if max_score == 0.0:
        max_diag = "D1"

    return {
        "diagnosis_code"   : max_diag,
        "diagnosis_label"  : DIAGNOSES[max_diag],
        "cf_score"         : round(max_score, 4),
        "cf_percentage"    : round(max_score * 100, 2),
        "all_scores"       : {k: round(v, 4) for k, v in cf_scores.items()},
    }

print("✅ Fungsi CF siap digunakan!")

# BAGIAN 3 — TRAINING MODEL ML
> Proses melatih model dengan dataset clean_dataset.csv
> Hasil training disimpan ke folder ml_model/

## 3.1 Load Dataset

In [ ]:
df = pd.read_csv('clean_dataset.csv')

print(f"✅ Dataset berhasil diload!")
print(f"   Jumlah data  : {len(df)} baris")
print(f"   Jumlah kolom : {len(df.columns)} kolom")
print(f"   Kolom        : {df.columns.tolist()}")
print()
print("Distribusi label Stress_Level:")
dist = df['Stress_Level'].value_counts().sort_index()
for level, count in dist.items():
    keterangan = {1:'Tidak Stres', 2:'Stres Ringan', 3:'Stres Sedang', 4:'Stres Berat', 5:'Stres Sangat Berat'}
    bar = '█' * (count // 30)
    print(f"  Level {level} ({keterangan[level]:18s}): {count} data  {bar}")

df.head()

## 3.2 EDA — Melihat Pola Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribusi label
df['Stress_Level'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Distribusi Tingkat Stres')
axes[0].set_xlabel('Stress Level')
axes[0].set_ylabel('Jumlah Data')
axes[0].tick_params(axis='x', rotation=0)

# Jam kerja per level stres
df.boxplot(column='Avg_Working_Hours_Per_Day', by='Stress_Level', ax=axes[1])
axes[1].set_title('Jam Kerja vs Tingkat Stres')
axes[1].set_xlabel('Stress Level')
plt.sca(axes[1]); plt.title('Jam Kerja vs Tingkat Stres')

# Korelasi
corr = df.corr()[['Stress_Level']].sort_values('Stress_Level', ascending=False)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=axes[2])
axes[2].set_title('Korelasi dengan Stress Level')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA selesai!")

## 3.3 Preprocessing

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df.drop('Stress_Level', axis=1)
y = df['Stress_Level'] - 1   # jadi 0,1,2,3,4

LABEL_NAMES = ['Tidak Stres', 'Stres Ringan', 'Stres Sedang', 'Stres Berat', 'Stres Sangat Berat']

# Split data 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ Preprocessing selesai!")
print(f"   Data training : {len(X_train)} baris")
print(f"   Data testing  : {len(X_test)} baris")

## 3.4 Training & Perbandingan Algoritma

In [ ]:
# ─── Training semua algoritma ───
models = {
    'Random Forest':       (RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42), False),
    'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42), False),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=8, random_state=42), False),
    'Logistic Regression': (LogisticRegression(max_iter=2000, random_state=42), True),
}

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("Training sedang berjalan... harap tunggu ⏳")
print()
print(f"{'Algoritma':25s} | {'CV Accuracy':>22s} | {'Test Acc':>10s}")
print("-" * 65)

for name, (model, scaled) in models.items():
    Xu = X_train_sc if scaled else X_train.values
    Xt = X_test_sc  if scaled else X_test.values
    cv = cross_val_score(model, Xu, y_train, cv=skf, scoring='accuracy')
    model.fit(Xu, y_train)
    test_acc = accuracy_score(y_test, model.predict(Xt))
    results[name] = {'cv': cv.mean(), 'cv_std': cv.std(), 'test': test_acc, 'model': model, 'scaled': scaled}
    print(f"{name:25s} | {cv.mean():.4f} ± {cv.std():.4f}           | {test_acc:.4f}")

print()
print("✅ Training selesai!")
print("📌 Random Forest dipilih sebagai model produksi")

## 3.5 Evaluasi Model

In [ ]:
best_model = results['Random Forest']['model']
y_pred     = best_model.predict(X_test.values)

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=LABEL_NAMES).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix — Random Forest')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Feature importances
fi = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importances')

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Simpan Model

In [ ]:
os.makedirs('ml_model', exist_ok=True)

joblib.dump(best_model,      'ml_model/stress_model.pkl')
joblib.dump(scaler,          'ml_model/scaler.pkl')
joblib.dump(list(X.columns), 'ml_model/feature_names.pkl')
joblib.dump(LABEL_NAMES,     'ml_model/label_names.pkl')

meta = {
    'model'        : 'RandomForestClassifier',
    'cv_accuracy'  : round(results['Random Forest']['cv'], 4),
    'test_accuracy': round(results['Random Forest']['test'], 4),
    'features'     : list(X.columns),
    'labels'       : LABEL_NAMES,
}
with open('ml_model/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("✅ Model berhasil disimpan di folder ml_model/")
print()
for fname in sorted(os.listdir('ml_model')):
    size = os.path.getsize(f'ml_model/{fname}')
    print(f"   📄 {fname:35s} ({size:,} bytes)")

# BAGIAN 4 — TESTING SISTEM
> Coba semua skenario untuk cek apakah hasilnya masuk akal

In [ ]:
# ─── Fungsi helper untuk ML dan Kombinasi ───

def predict_ml(data_dict):
    loaded_model    = joblib.load('ml_model/stress_model.pkl')
    loaded_labels   = joblib.load('ml_model/label_names.pkl')
    loaded_features = joblib.load('ml_model/feature_names.pkl')
    row   = np.array([[data_dict[f] for f in loaded_features]])
    proba = loaded_model.predict_proba(row)[0]
    idx   = np.argmax(proba)
    return {"label": loaded_labels[idx], "confidence": proba[idx], "proba": dict(zip(loaded_labels, proba))}

def combine_cf_ml(cf_result, ml_result, cf_weight=0.7):
    ml_weight  = 1 - cf_weight
    cf_scores  = cf_result['all_scores']
    cf_vec     = np.array([cf_scores.get(f'D{i+1}', 0.0) for i in range(4)])
    proba_list = list(ml_result['proba'].values())
    ml_vec4    = np.array([proba_list[0], proba_list[1], proba_list[2], proba_list[3]+proba_list[4]])
    cf_norm    = cf_vec  / cf_vec.sum()  if cf_vec.sum()  > 0 else cf_vec
    ml_norm    = ml_vec4 / ml_vec4.sum() if ml_vec4.sum() > 0 else ml_vec4
    combined   = cf_weight * cf_norm + ml_weight * ml_norm
    labels4    = ['Tidak Stres', 'Stres Ringan', 'Stres Sedang', 'Stres Berat']
    final_idx  = int(np.argmax(combined))
    return {"final_diagnosis": labels4[final_idx], "final_code": f"D{final_idx+1}",
            "final_score": round(float(combined[final_idx]), 4),
            "score_breakdown": dict(zip(labels4, combined.round(4).tolist()))}

REKOMENDASI = {
    "D1": {"warna": "🟢", "summary": "Kondisi baik. Pertahankan keseimbangan kerja dan kehidupan.",
           "tips": ["Pertahankan kebiasaan tidur dan olahraga", "Jaga komunikasi positif dengan tim"]},
    "D2": {"warna": "🟡", "summary": "Stres ringan. Perlu beberapa penyesuaian kecil.",
           "tips": ["Coba teknik manajemen waktu (Pomodoro)", "Pastikan tidur 7-8 jam/malam", "Olahraga 30 menit/hari"]},
    "D3": {"warna": "🟠", "summary": "Stres sedang. Perlu tindakan lebih serius.",
           "tips": ["Diskusikan beban kerja ke HR/atasan", "Coba meditasi atau teknik pernapasan", "Pertimbangkan konsultasi konselor"]},
    "D4": {"warna": "🔴", "summary": "Stres berat. Segera ambil langkah penanganan.",
           "tips": ["Konsultasi dengan psikolog atau dokter", "Ajukan cuti atau kurangi beban kerja", "Jangan tangani sendiri"]},
}

print("✅ Fungsi helper siap!")

## 4.1 Test 5 Skenario Karyawan

In [ ]:
skenario = [
    {
        "nama": "👤 Karyawan A — Kondisi Normal",
        "cf" : {"G7": 0.25, "G8": 0.25, "G11": 0.0, "G18": 0.0},
        "ml" : {"Avg_Working_Hours_Per_Day": 7.0, "Work_From": 2, "Work_Pressure": 1,
                "Manager_Support": 5, "Sleeping_Habit": 5, "Exercise_Habit": 4,
                "Job_Satisfaction": 5, "Work_Life_Balance": 1, "Social_Person": 4, "Lives_With_Family": 1},
        "harapan": "Tidak Stres"
    },
    {
        "nama": "👤 Karyawan B — Stres Ringan",
        "cf" : {"G1": 0.75, "G3": 0.75, "G5": 0.75, "G10": 0.5, "G23": 1.0},
        "ml" : {"Avg_Working_Hours_Per_Day": 10.0, "Work_From": 0, "Work_Pressure": 3,
                "Manager_Support": 3, "Sleeping_Habit": 3, "Exercise_Habit": 2,
                "Job_Satisfaction": 3, "Work_Life_Balance": 1, "Social_Person": 3, "Lives_With_Family": 0},
        "harapan": "Stres Ringan"
    },
    {
        "nama": "👤 Karyawan C — Stres Sedang",
        "cf" : {"G2": 0.75, "G3": 1.0, "G15": 0.75, "G16": 1.0, "G19": 0.75, "G20": 0.75},
        "ml" : {"Avg_Working_Hours_Per_Day": 12.0, "Work_From": 0, "Work_Pressure": 4,
                "Manager_Support": 2, "Sleeping_Habit": 2, "Exercise_Habit": 1,
                "Job_Satisfaction": 2, "Work_Life_Balance": 0, "Social_Person": 2, "Lives_With_Family": 0},
        "harapan": "Stres Sedang"
    },
    {
        "nama": "👤 Karyawan D — Stres Berat",
        "cf" : {"G2": 1.0, "G3": 1.0, "G16": 1.0, "G17": 1.0, "G30": 1.0, "G33": 1.0, "G39": 0.75, "G43": 1.0},
        "ml" : {"Avg_Working_Hours_Per_Day": 14.5, "Work_From": 0, "Work_Pressure": 5,
                "Manager_Support": 1, "Sleeping_Habit": 1, "Exercise_Habit": 1,
                "Job_Satisfaction": 1, "Work_Life_Balance": 0, "Social_Person": 1, "Lives_With_Family": 0},
        "harapan": "Stres Berat"
    },
]

print("=" * 65)
print("  HASIL TEST — 4 SKENARIO KARYAWAN")
print("=" * 65)

for sk in skenario:
    cf_result = diagnose_cf(sk["cf"])
    ml_result = predict_ml(sk["ml"])
    final     = combine_cf_ml(cf_result, ml_result)
    rek       = REKOMENDASI[final["final_code"]]
    cocok     = "✅" if sk["harapan"] in final["final_diagnosis"] else "⚠️"

    print(f"\n{sk['nama']}")
    print(f"  Harapan      : {sk['harapan']}")
    print(f"  CF Diagnosis : {cf_result['diagnosis_label']} (score: {cf_result['cf_score']:.3f})")
    print(f"  ML Prediksi  : {ml_result['label']} (conf: {ml_result['confidence']:.2%})")
    print(f"  {cocok} FINAL: {final['final_diagnosis']} (score: {final['final_score']:.4f})")
    print(f"  {rek['warna']} {rek['summary']}")

# BAGIAN 5 — COBA INPUT KAMU SENDIRI ✍️
> Edit nilai di bawah sesuai kondisi yang ingin kamu uji
> lalu jalankan cell ini

In [ ]:
# ══════════════════════════════════════════════════════════
# ✏️ EDIT BAGIAN INI SESUAI KONDISI YANG INGIN DIUJI
# ══════════════════════════════════════════════════════════

# Skala jawaban CF:
# 0.0  = Tidak Pernah
# 0.25 = Jarang
# 0.5  = Kadang-kadang
# 0.75 = Sering
# 1.0  = Selalu

MY_CF_ANSWERS = {
    # ── Beban & Tekanan Kerja ──
    "G1" : 0.75,   # Tugas terasa berlebihan
    "G2" : 0.5,    # Tanggung jawab memberatkan
    "G3" : 0.75,   # Dikejar deadline
    "G5" : 0.5,    # Sulit memenuhi target
    "G6" : 0.5,    # Waktu istirahat kurang

    # ── Konflik Peran ──
    "G16": 0.5,    # Tekanan dari atasan

    # ── Hubungan Interpersonal ──
    "G19": 0.25,   # Konflik dengan rekan

    # ── Kepemimpinan ──
    "G32": 0.5,    # Tidak tahu penilaian atasan

    # ── Karir ──
    "G37": 0.5,    # Peluang promosi kecil
    "G42": 0.5,    # Feedback tidak sesuai
}

MY_ML_FEATURES = {
    "Avg_Working_Hours_Per_Day": 9.0,   # rata-rata jam kerja per hari
    "Work_From"               : 1,       # 0=Kantor | 1=WFH | 2=Hybrid
    "Work_Pressure"           : 3,       # 1(ringan) - 5(sangat berat)
    "Manager_Support"         : 3,       # 1(tidak ada) - 5(penuh)
    "Sleeping_Habit"          : 3,       # 1(sangat buruk) - 5(sangat baik)
    "Exercise_Habit"          : 2,       # 1(tidak pernah) - 5(setiap hari)
    "Job_Satisfaction"        : 3,       # 1(sangat tidak puas) - 5(sangat puas)
    "Work_Life_Balance"       : 1,       # 0=Tidak seimbang | 1=Seimbang
    "Social_Person"           : 3,       # 1(sangat tertutup) - 5(mudah bergaul)
    "Lives_With_Family"       : 1,       # 0=Sendiri | 1=Bersama keluarga
}

# ══════════════════════════════════════════════════════════
# JANGAN EDIT DI BAWAH INI
# ══════════════════════════════════════════════════════════

cf_result = diagnose_cf(MY_CF_ANSWERS)
ml_result = predict_ml(MY_ML_FEATURES)
final     = combine_cf_ml(cf_result, ml_result)
rek       = REKOMENDASI[final["final_code"]]

print("╔══════════════════════════════════════════════╗")
print("║         HASIL DIAGNOSIS STRES KAMU          ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  CF Expert System : {cf_result['diagnosis_label']:25s}║")
print(f"║  CF Score         : {cf_result['cf_score']:.4f} ({cf_result['cf_percentage']:.1f}%)         ║")
print(f"║  ML Prediksi      : {ml_result['label']:25s}║")
print(f"║  ML Confidence    : {ml_result['confidence']:.2%}                      ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  {rek['warna']} FINAL: {final['final_diagnosis']:37s}║")
print("╚══════════════════════════════════════════════╝")
print()
print(f"📋 {rek['summary']}")
print()
print("💡 Rekomendasi:")
for tip in rek['tips']:
    print(f"   ✓ {tip}")
print()
print("📊 Score Breakdown:")
for lbl, score in final['score_breakdown'].items():
    bar  = '█' * int(score * 30)
    mark = " ◄ FINAL" if lbl == final['final_diagnosis'] else ""
    print(f"   {lbl:15s}: {score:.4f}  {bar}{mark}")

# BAGIAN 6 — DOWNLOAD MODEL (Opsional)
> Jalankan ini kalau ingin download file model ke komputer kamu
> untuk dipakai di backend API

In [ ]:
# Download folder ml_model sebagai zip
from google.colab import files
import shutil

shutil.make_archive('ml_model', 'zip', 'ml_model')
files.download('ml_model.zip')
print("✅ Download dimulai!")
print("   Setelah didownload, extract dan taruh di folder backend kamu.")